# Impulse — Getting Started

The simplest possible Impulse report: **one signal, one event, one histogram** — using only the **three required silver tables**.

That is the whole point of this notebook: you do **not** need tag tables, channel mappings, or unit-conversion tables to get value out of Impulse. Three tables is enough.

| Required table | What it holds |
|----------------|---------------|
| `container_metrics` | one row per recording — `container_id`, timestamps, and any metadata columns you want to carry through |
| `channel_metrics` | one row per signal — includes a **`channel_name`** column used to select signals by name |
| `channels` | the raw `(timestamp, value)` samples |

Run the cells top to bottom on a serverless Databricks cluster. Fill in the **Catalog**, **Schema**, and **Table Prefix** widgets that appear after running cell 1.

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text("table_prefix", "", "Table Prefix")

In [ ]:
import sys, os
import pandas as pd

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE_PREFIX = dbutils.widgets.get("table_prefix")

if not CATALOG or not SCHEMA or not TABLE_PREFIX:
    raise ValueError("Please set Catalog, Schema, and Table Prefix widgets above before running.")

nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
DEMOS_DIR = "/Workspace" + "/".join(nb_path.split("/")[:-1])
REPO_ROOT = "/Workspace" + "/".join(nb_path.split("/")[:-2])
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# Only THREE tables are required to run Impulse.
csv_dir = os.path.join(DEMOS_DIR, "data", "getting_started")
for t in ["container_metrics", "channel_metrics", "channels"]:
    (spark.createDataFrame(pd.read_csv(f"{csv_dir}/{t}.csv"))
          .write.mode("overwrite")
          .saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_{t}"))
print(f"Loaded 3 silver-layer tables under {CATALOG}.{SCHEMA}.{TABLE_PREFIX}_*")

## Define and run the report

Without tag tables, signals are selected by a **`channel_name` column on `channel_metrics`**:

```python
db.query.channel(channel_name="Engine RPM")
```

That is the only difference from the tag-based setup. The config below lists just the three source tables — `container_tags_table` and `channel_tags_table` are simply omitted, which puts the `DefaultSolver` into wide mode.

In [ ]:
from databricks.sdk import WorkspaceClient

from impulse_reporting.aggregations.histogram import HistogramDuration
from impulse_reporting.core.page import Page
from impulse_reporting.core.report import Report
from impulse_reporting.events.basic_event import BasicEvent

pfx = f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}"

# Three source tables — no tag tables, no channel mapping, no unit conversion.
config = {
    "source": {
        "container_metrics_table": f"{pfx}_container_metrics",
        "channel_metrics_table":   f"{pfx}_channel_metrics",
        "channels_uri":            f"{pfx}_channels",
    },
    "unity_sink": {
        "catalog": CATALOG,
        "schema":  SCHEMA,
        "table_prefix": TABLE_PREFIX,
    },
    "query_engine": {"solver": "DefaultSolver", "data_type": "RAW"},
    "measurement_dimensions": ["container_id", "vehicle_key", "start_ts", "stop_ts"],
}

report = Report(
    name="getting_started",
    spark=spark,
    workspace_client=WorkspaceClient(),
    config=config,
)
db = report.get_db()

# Select a signal by its channel_name column on channel_metrics (no tag tables needed).
eng_rpm = db.query.channel(channel_name="Engine RPM")

# One event: engine RPM above 2000.
high_rpm = BasicEvent(
    name="high_rpm",
    expr=eng_rpm > 2000,
    desc="Engine RPM above 2000",
)
report.add_event(high_rpm)

# One duration-weighted histogram of RPM, scoped to the event.
page = Page(page_number=1)
page.add_aggregation(
    HistogramDuration(
        name="rpm_distribution",
        base_expr=eng_rpm,
        bins=[float(x) for x in range(0, 5001, 500)],
        event=high_rpm,
        channel_name="Engine RPM",
        bins_unit="rpm",
        values_unit="s",
    )
)
report.add_page(page)

report.determine_report()
report.persist_results()
print(f"Report persisted to {pfx}_*")

In [ ]:
import pyspark.sql.functions as F

# Duration spent in each RPM bin (summed across all recordings).
display(
    spark.read.table(f"{pfx}_histogram_fact")
    .groupBy("bin_name", "lower_bound")
    .agg(F.sum("hist_value").alias("duration"))
    .orderBy("lower_bound")
)